In [1]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('../5 月/0510/NES_new_vserison/')
from NES_VMC import NESTotalAnsatz, create_machine, create_machine_matrix, create_single_machine, \
    ha, SingleStateAnsatz, NESFermionHopRule, compute_qgt, nes_vmc_gradient
import optax
from jax.flatten_util import ravel_pytree
import time

print('='*60)
print('LiH NES-VMC 计算前三个能级 (K=3)')
print('='*60)

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: Prefer the new nk.driver.VMC_SR over VMC which supports minSR and SPRING.

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV
LiH NES-VMC 计算前三个能级 (K=3)


## 1. LiH 分子定义与 FCI 基准

In [2]:
from pyscf import gto, scf, fci

# LiH 分子几何结构
# Li-H 键长 ~1.595 Å
bond_length = 1.595
geometry = [
    ('Li', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.))
]

# 创建分子对象，使用 STO-3G 基组
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

# 计算 FCI 精确基准能量
cisolver = fci.FCI(mf)
cisolver.nroots = 4  # 计算前4个态用于参考
E_fcis, fcivec = cisolver.kernel()

print('='*60)
print('LiH FCI 基准能量')
print('='*60)
for i, e in enumerate(E_fcis[:3]):
    exc = (e - E_fcis[0]) * 27.2114
    print(f'E{i} = {e:.8f} Ha  |  激发能：{exc:.4f} eV')

# 分子轨道信息
n_orb = mol.nao_nr()
n_elec = mol.nelectron
print(f'\n轨道数: {n_orb}, 电子数: {n_elec}')

LiH FCI 基准能量
E0 = -7.88240193 Ha  |  激发能：0.0000 eV
E1 = -7.76641848 Ha  |  激发能：3.1561 eV
E2 = -7.74921619 Ha  |  激发能：3.6242 eV

轨道数: 6, 电子数: 4


In [3]:
import itertools
import netket as nk

# 1. 给定 HF 参考态
hf_state = [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1]

# 2. 提取占据轨道、空轨道
occ = [idx for idx, val in enumerate(hf_state) if val == 1]
virt = [idx for idx, val in enumerate(hf_state) if val == 0]

print("HF 占据轨道 occ =", occ)
print("HF 空轨道 virt =", virt)

# 3. 生成 CCSD 对应的跃迁边（集合自动去重）
edges_set = set()

# -------- 单激发 (i -> a) --------
for i in occ:
    for a in virt:
        edges_set.add((i, a))

# -------- 双激发 (i,j -> a,b) 拆为两条单跃迁 --------
for i, j in itertools.combinations(occ, 2):
    for a, b in itertools.combinations(virt, 2):
        edges_set.add((i, a))
        edges_set.add((j, b))

# 转为有序列表
edges = sorted(list(edges_set))
print(f"总边数: {len(edges)}")
print("edges =", edges)

HF 占据轨道 occ = [4, 5, 10, 11]
HF 空轨道 virt = [0, 1, 2, 3, 6, 7, 8, 9]
总边数: 32
edges = [(4, 0), (4, 1), (4, 2), (4, 3), (4, 6), (4, 7), (4, 8), (4, 9), (5, 0), (5, 1), (5, 2), (5, 3), (5, 6), (5, 7), (5, 8), (5, 9), (10, 0), (10, 1), (10, 2), (10, 3), (10, 6), (10, 7), (10, 8), (10, 9), (11, 0), (11, 1), (11, 2), (11, 3), (11, 6), (11, 7), (11, 8), (11, 9)]


## 2. 希尔伯特空间与哈密顿量设置

In [6]:
# 从 PySCF 分子创建 NetKet 哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

# LiH: 4电子 (2个自旋向上, 2个自旋向下)
# 在 STO-3G 基组下有 6 个轨道
n_orbitals = mol.nao_nr()
n_alpha = n_elec // 2 + n_elec % 2  # 2
n_beta = n_elec // 2  # 2

print(f'轨道数: {n_orbitals}')
print(f'Alpha电子数: {n_alpha}, Beta电子数: {n_beta}')

# 希尔伯特空间设置
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=n_orbitals,
    s=1/2,
    n_fermions_per_spin=(n_alpha, n_beta)
)
print(f'希尔伯特空间维度: {hi.n_states}')

# NES 扩展副本数 K=3 (计算前3个能级: 基态 + 2个激发态)
K = 3
hi_ext = hi ** K
SINGLE_SIZE = hi.size
print(f'扩展希尔伯特空间维度: {hi_ext.n_states}')

轨道数: 6
Alpha电子数: 2, Beta电子数: 2
希尔伯特空间维度: 225
扩展希尔伯特空间维度: 11390625


## 3. 神经网络波函数 (Ansatz) 初始化

In [7]:
# 创建 NES Total Ansatz
hidden_dim = 32
total_ansatz = NESTotalAnsatz(
    n_spin_orbitals=n_orbitals * 2,  # 考虑自旋
    n_states=K,
    hidden_dim=hidden_dim,
    rngs=nnx.Rngs(42)
)

# 创建总机器和矩阵机器
total_machine, total_graphdef, total_params = create_machine(total_ansatz)
total_matrix_machine, _, _ = create_machine_matrix(total_ansatz)

# 创建单个态的机器列表
single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)

print(f'参数总数: {ravel_pytree(total_params)[0].shape[0]}')

参数总数: 4515


## 4. NES 采样器设置

In [8]:
# 定义费米子跃迁边 (根据轨道生成)
single_edges = []
for i in range(n_orbitals):
    for j in range(i+1, n_orbitals):
        # 每个轨道对 (i, j) 对应两个自旋通道
        single_edges.append((i, j))
        single_edges.append((i + n_orbitals, j + n_orbitals))

single_edges = tuple(single_edges)
print(f'跃迁边数: {len(single_edges)}')

# 构建扩展空间的跃迁边
ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)

# 创建 NES 采样规则
nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)

# 创建采样器
N_CHAINS = 16
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=N_CHAINS,
    sweep_size=20
)

print(f'采样器: {N_CHAINS} 链, 每链采样长度 200, 热身 100')

跃迁边数: 30
采样器: 16 链, 每链采样长度 200, 热身 100


## 5. 优化器设置

In [9]:
# 优化器设置
optimizer = optax.sgd(learning_rate=0.002)
opt_state = optimizer.init(total_params)

# 训练超参数
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
N_ITER = 300

print(f'训练参数: 学习率=0.002, 迭代={N_ITER}, K={K}')

训练参数: 学习率=0.002, 迭代=300, K=3


## 6. NES-VMC 训练循环

In [10]:
# 初始化采样器状态
sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# 训练历史记录
history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_2st': [],
    'loss': [],
    'grad_norm': [],
    'log_Psi_mean': [],
}

print('\n' + '='*60)
print('开始 NES-VMC 训练 (K=3)')
print('='*60)
print(f'FCI 基态能量: {E_fcis[0]:.8f} Ha')
print(f'FCI 第一激发态: {E_fcis[1]:.8f} Ha (激发能: {(E_fcis[1]-E_fcis[0])*27.2114:.4f} eV)')
print(f'FCI 第二激发态: {E_fcis[2]:.8f} Ha (激发能: {(E_fcis[2]-E_fcis[0])*27.2114:.4f} eV)')
print('='*60 + '\n')

start_time = time.time()

for step in range(N_ITER):
    # 1. 采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine,
        parameters=total_params,
        state=sampler_state,
        chain_length=N_SAMPLES_PER_CHAIN
    )
    
    # 2. 维度重塑
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, SINGLE_SIZE)
    
    # 3. 计算梯度
    grad, loss_mean, E_L_mean = nes_vmc_gradient(
        ha=ha,
        total_matrix_machine=total_matrix_machine,
        total_machine=total_machine,
        single_machine_list=single_machine_list,
        total_params=total_params,
        x_batch=x_batch
    )
    
    # 4. 计算自然梯度
    grad_flat, grad_unravel_fn = ravel_pytree(grad)
    qgt_reg, _ = compute_qgt(total_machine, total_params, x_batch.reshape(-1, K, n_orbitals*2), diag_shift=0.1)
    
    natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad_flat)
    grad = natural_grad
    
    # 5. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    
    # 6. 记录结果
    eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
    grad_norm = jnp.linalg.norm(grad_flat)
    
    log_Psi_batch = total_machine(total_params, x_batch)
    
    history['step'].append(step)
    history['loss'].append(loss_mean)
    history['grad_norm'].append(grad_norm)
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['energy_2st'].append(eig_vals[2])
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    
    # 7. 打印进度
    if step % 30 == 0 or step == N_ITER - 1:
        print(f"Step {step:4d} | Loss: {loss_mean:.6f} | "
              f"E0={eig_vals[0]:.6f} Ha | E1={eig_vals[1]:.6f} Ha | E2={eig_vals[2]:.6f} Ha | "
              f"grad_norm={grad_norm:.4f}")

end_time = time.time()
print(f'\n训练耗时: {end_time - start_time:.2f} 秒')


开始 NES-VMC 训练 (K=3)
FCI 基态能量: -7.88240193 Ha
FCI 第一激发态: -7.76641848 Ha (激发能: 3.1561 eV)
FCI 第二激发态: -7.74921619 Ha (激发能: 3.6242 eV)

Step    0 | Loss: 20.996158 | E0=-345.087356 Ha | E1=-5.555576 Ha | E2=371.639090 Ha | grad_norm=1094960.5034
Step   30 | Loss: nan | E0=nan Ha | E1=nan Ha | E2=nan Ha | grad_norm=nan


KeyboardInterrupt: 

## 7. 结果分析

In [ ]:
# 取最后几个迭代的平均值作为最终结果
n_avg = 20
final_E0 = jnp.mean(jnp.array(history['energy_0st'][-n_avg:]))
final_E1 = jnp.mean(jnp.array(history['energy_1st'][-n_avg:]))
final_E2 = jnp.mean(jnp.array(history['energy_2st'][-n_avg:]))

print('='*60)
print('NES-VMC 计算结果 (K=3)')
print('='*60)
print(f'基态能量 E0 = {final_E0:.8f} Ha (FCI: {E_fcis[0]:.8f} Ha, 误差: {(final_E0-E_fcis[0])*1000:.4f} mHa)')
print(f'第一激发态 E1 = {final_E1:.8f} Ha (FCI: {E_fcis[1]:.8f} Ha, 误差: {(final_E1-E_fcis[1])*1000:.4f} mHa)')
print(f'第二激发态 E2 = {final_E2:.8f} Ha (FCI: {E_fcis[2]:.8f} Ha, 误差: {(final_E2-E_fcis[2])*1000:.4f} mHa)')
print('='*60)
print(f'\n激发能:')
print(f'NES-VMC: E1-E0 = {(final_E1-final_E0)*27.2114:.4f} eV, E2-E0 = {(final_E2-final_E0)*27.2114:.4f} eV')
print(f'FCI:     E1-E0 = {(E_fcis[1]-E_fcis[0])*27.2114:.4f} eV, E2-E0 = {(E_fcis[2]-E_fcis[0])*27.2114:.4f} eV')

In [ ]:
# 绘制训练曲线
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 基态能量
ax = axes[0, 0]
ax.plot(history['energy_0st'], label='NES-VMC E0', color='blue')
ax.axhline(E_fcis[0], color='red', linestyle='--', label=f'FCI E0 = {E_fcis[0]:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Energy (Ha)')
ax.set_title('Ground State Energy')
ax.legend()
ax.grid(True, alpha=0.3)

# 第一激发态
ax = axes[0, 1]
ax.plot(history['energy_1st'], label='NES-VMC E1', color='blue')
ax.axhline(E_fcis[1], color='red', linestyle='--', label=f'FCI E1 = {E_fcis[1]:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Energy (Ha)')
ax.set_title('First Excited State Energy')
ax.legend()
ax.grid(True, alpha=0.3)

# 第二激发态
ax = axes[1, 0]
ax.plot(history['energy_2st'], label='NES-VMC E2', color='blue')
ax.axhline(E_fcis[2], color='red', linestyle='--', label=f'FCI E2 = {E_fcis[2]:.4f}')
ax.set_xlabel('Iteration')
ax.set_ylabel('Energy (Ha)')
ax.set_title('Second Excited State Energy')
ax.legend()
ax.grid(True, alpha=0.3)

# Loss
ax = axes[1, 1]
ax.plot(history['loss'], label='Loss', color='green')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.set_title('NES-VMC Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('LiH_NES_VMC_K3_results.png', dpi=150)
plt.show()

print('图像已保存为 LiH_NES_VMC_K3_results.png')

In [ ]:
# 保存训练历史
import pickle
import os

os.makedirs('./data', exist_ok=True)
with open('./data/LiH_NES_VMC_K3_history.pkl', 'wb') as f:
    pickle.dump(history, f)

print('训练历史已保存为 ./data/LiH_NES_VMC_K3_history.pkl')